# SRHarness backbone performance correlates

Compare the five complete LSR-Synth backbone runs against long-horizon agent benchmarks, activated parameter count, release date, and GPQA Diamond. The LSR-Synth aggregates below are the problem-count-weighted overall results from `baseline/符号回归智能体-实验结果.xlsx` (44 Physics, 36 Chemistry, 24 Biology, and 25 Material problems). External model attributes match the sources cited in the paper appendix.

In [ ]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from yumeow_plot import get_fig

output_dir = Path('./logs/analysis/260924模型性能对比分析')
output_dir.mkdir(parents=True, exist_ok=True)

df = pd.DataFrame({
    'model': [
        'DeepSeek-v4-flash-0731', 'DeepSeek-v4-pro-0813',
        'DeepSeek-v4.1-flash', 'GLM-5.3-flash', 'GLM-5.3',
    ],
    'short': ['DS-F', 'DS-P', 'DS-4.1F', 'GLM-F', 'GLM'],
    'id_acc': [91.5098, 94.9968, 95.5209, 92.5550, 93.4682],
    'ood_acc': [83.4202, 87.8481, 92.4078, 86.6434, 86.9628],
    'sa': [6.2016, 11.6279, 15.5039, 10.8527, 10.8527],
    'terminalbench': [82.7, 87.9, 90.6, 84.3, 88.2],
    'deepswe': [54.4, 62.7, 74.2, 63.4, 66.9],
    # Activated parameters per token; v4.1-flash uses its decoding value.
    'active_params_b': [13, 49, 16, 18, 40],
    'release_date': pd.to_datetime([
        '2026-07-31', '2026-08-13', '2026-09-10', '2026-08-26', '2026-08-14',
    ]),
    'gpqa_diamond': [89.9, 92.4, 90.9, 91.2, 88.1],
})

# Min-max each agent benchmark before averaging so their different scales
# do not give either benchmark disproportionate weight.
for col in ('terminalbench', 'deepswe'):
    lo, hi = df[col].min(), df[col].max()
    df[f'{col}_scaled'] = 100 * (df[col] - lo) / (hi - lo)
df['agentic_composite'] = df[['terminalbench_scaled', 'deepswe_scaled']].mean(axis=1)
df


In [ ]:
fig, axes = get_fig(RN=2, CN=3, FW=28, A_ratio=1, HS=3, VS=7, LM=5, RM=5, TM=1.5, BM=5, dpi=300, fontsize=14, lw=1)
ax_loader = iter(axes)

metric_specs = [
    ('id_acc', r'ID $\mathrm{Acc}_{0.1}$', '#d33f6a', 'o'),
    ('ood_acc', r'OOD $\mathrm{Acc}_{0.1}$', '#4a6fe3', 's'),
    ('sa', 'SA', '#bb65b6', '^'),
]
panel_specs = [
    ('agentic_composite', 'Long-horizon agent\nbenchmark composite'),
    ('terminalbench', 'TerminalBench 2.1'),
    ('deepswe', 'DeepSWE'),
    ('active_params_b', 'Activated parameters (B)'),
    ('release_date', 'Release date (2026)'),
    ('gpqa_diamond', 'GPQA Diamond'),
]

legend_handles = []
for panel_idx, (x_col, x_label) in enumerate(panel_specs):
    ax = next(ax_loader)
    ax_sa = ax.twinx()
    order = np.argsort(df[x_col].to_numpy())
    x = df[x_col].to_numpy()[order]

    for metric, label, color, marker in metric_specs[:2]:
        y = df[metric].to_numpy()[order]
        line, = ax.plot(x, y, color=color, marker=marker, markersize=6,
                        linewidth=1, label=label, zorder=3)
        if panel_idx == 0:
            legend_handles.append(line)

    y_sa = df['sa'].to_numpy()[order]
    line_sa, = ax_sa.plot(x, y_sa, color=metric_specs[2][2], marker=metric_specs[2][3],
                          markersize=6, linewidth=1, label='SA', zorder=3)
    if panel_idx == 0:
        legend_handles.append(line_sa)

    ax.set_xlabel(x_label, fontsize=14)
    ax.set_ylim(80, 100)
    ax_sa.set_ylim(0, 20)
    ax.set_yticks([80, 85, 90, 95, 100])
    ax_sa.set_yticks([0, 5, 10, 15, 20])
    ax.tick_params(axis='both', labelsize=10)
    ax_sa.tick_params(axis='y', labelsize=10, colors=metric_specs[2][2])
    ax.grid(True, color='#d9d9d9', linewidth=1, alpha=0.7, zorder=0)

    if panel_idx % 3 == 0:
        ax.set_ylabel(r'Accuracy (\%)', fontsize=14)
    else:
        ax.tick_params(axis='y', labelleft=False)
    if panel_idx % 3 == 2:
        ax_sa.set_ylabel(r'SA (\%)', fontsize=14, color=metric_specs[2][2])
    else:
        ax_sa.tick_params(axis='y', labelright=False)

    x_for_corr = (df[x_col].astype('int64') if x_col == 'release_date' else df[x_col])
    rho = [spearmanr(x_for_corr, df[m]).statistic for m, *_ in metric_specs]
    rho_text = rf'$\rho$ (ID/OOD/SA): {rho[0]:.2f}/{rho[1]:.2f}/{rho[2]:.2f}'
    ax.text(0.5, 0.025, rho_text, transform=ax.transAxes, fontsize=10,
            ha='center', va='bottom',
            bbox={'boxstyle': 'round,pad=0.25', 'facecolor': 'white',
                  'edgecolor': '#bdbdbd', 'alpha': 0.88})

    if x_col == 'release_date':
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
        ax.tick_params(axis='x', rotation=30)
    if panel_idx == 0:
        label_offsets = {'DS-F': (3, 5), 'DS-P': (-14, -16), 'DS-4.1F': (-43, 5),
                         'GLM-F': (-7, 7), 'GLM': (3, -16)}
        for _, row in df.iterrows():
            ax.annotate(row['short'], (row[x_col], row['id_acc']),
                        xytext=label_offsets[row['short']], textcoords='offset points',
                        fontsize=10, color='#333333')

fig.legend(legend_handles, [h.get_label() for h in legend_handles],
           loc='lower center', bbox_to_anchor=(0.5, 0.995), ncol=3,
           frameon=False, fontsize=10)

stem = output_dir / 'lsrsynth_backbone_performance_correlates'
for ext in ('png', 'svg', 'pdf'):
    fig.savefig(f'{stem}.{ext}', dpi=300, bbox_inches='tight')
fig
